### Imports & Downloads

In [44]:
%pip install uv --quiet
%uv pip install pandas numpy plotly matplotlib
%uv sync

Note: you may need to restart the kernel to use updated packages.


c:\Users\specb\Desktop\School\csc5260\project\Modern-Store-Of-Value\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.


Using Python 3.11.15 environment at: C:\Users\specb\Desktop\School\csc5260\project\Modern-Store-Of-Value\.venv
Audited 4 packages in 18ms


Note: you may need to restart the kernel to use updated packages.


Resolved 132 packages in 3ms
Audited 128 packages in 21ms


In [45]:
# Data manipulation tools
import pandas as pd
import datetime

# Visualization tools
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# OS tools
from pathlib import Path
import sys

# Custom Stooq data importer
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.stooq_processor import StooqProcessor

### Downloads

In [46]:
stooq_tickers = {
    "Crypto ETFs": [
        "BITW",  # Bitwise 10 Crypto Index
        "IBIT",  # iShares Bitcoin Trust (Replaces BTC-USD)
        "ETHA"   # iShares Ethereum Trust (Replaces ETH-USD)
    ], 
    
    "Individual Stocks": [
        "NVDA", "AAPL", "MSFT", "AMD", "AMZN", "TSLA", "WMT", "LOW", "HD", "JNJ"
    ],
    
    "Sector ETFs": [
        "XLU"    # Utilities Select Sector SPDR Fund
    ],
    
    "Broad Market ETFs": [
        "SPY",   # S&P 500
        "VTI"   # Total US Market (replaces Wilshire 5000)
    ],
    
    "Commodity ETFs (Metals)": [
        "GLD",   # Gold (Baseline)
        "SLV",   # Silver
        "PPLT",  # Platinum
        "PALL"   # Palladium
    ],
    
    "Commodity ETFs (Agriculture)": [
        "WEAT",  # Wheat
        "SOYB",  # Soybeans
        "DBA"    # Broad Agriculture
    ]
}

start_date = "2021-01-01"
end_date = "2026-03-31"

In [60]:
# -----------------------------
# Flatten tickers + category map
# -----------------------------
category_map = {
    ticker: category
    for category, tickers in stooq_tickers.items()
    for ticker in tickers
}

flat_tickers = list(category_map.keys())


# -----------------------------
# Download data
# -----------------------------
with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:

    # Stores valid tickers
    valid_tickers = [
        t for t in flat_tickers
        if processor.has_ticker(t)
    ]

    # Prints tickers that were not valid
    missing = sorted(set(flat_tickers) - set(valid_tickers))
    if missing:
        print("Skipping missing tickers:", missing)

    data = processor.download(
        valid_tickers,
        start=start_date,
        end=end_date,
    )


# -----------------------------
# Attach metadata
# -----------------------------
for ticker, frame in data.items():
    data[ticker] = frame.assign(
        Ticker=ticker,
        Category=category_map[ticker],
    )


# -----------------------------
# Combine dataset
# -----------------------------
combined_data = pd.concat(data.values()).reset_index()

# data['AAPL'].tail()
# data["BITW"]
# combined_data[combined_data['Ticker'] == 'BITW']
combined_data

,Date,Open,High,Low,Close,Volume,OpenInt,Ticker,Category
0,2025-12-31,62.8200,63.1200,56.5600,58.7600,1752680.0,0,BITW,Crypto ETFs
1,2026-01-31,59.9400,66.4800,54.5000,55.6601,2425192.0,0,BITW,Crypto ETFs
2,2026-02-28,51.2878,52.2950,40.6599,43.0400,4596906.0,0,BITW,Crypto ETFs
3,2026-03-26,42.9952,49.4500,42.9952,44.8700,1583267.0,0,BITW,Crypto ETFs
4,2024-01-31,26.4000,26.4100,22.0200,24.3000,207876989.0,0,IBIT,Crypto ETFs
...,...,...,...,...,...,...,...,...,...
1307,2025-11-30,26.6000,26.9400,25.5500,26.4200,3917732.0,0,DBA,Commodity ETFs (Agriculture)
1308,2025-12-31,26.3600,26.6666,25.4000,25.5200,4877035.0,0,DBA,Commodity ETFs (Agriculture)
1309,2026-01-31,25.5000,26.0500,25.4250,25.6600,5404587.0,0,DBA,Commodity ETFs (Agriculture)
1310,2026-02-28,25.5500,26.1500,25.5400,26.0200,5422359.0,0,DBA,Commodity ETFs (Agriculture)


In [ ]:
# Debugging

# with StooqProcessor(repo_root / "data" / "d_us_txt.zip") as processor:
#     for ticker in flat_tickers:
#         if processor.has_ticker(ticker):
#             info = processor.get_ticker_info(ticker)
#             df = processor._read_daily_member(info)
#             print(ticker, df.shape[0])

### Calculate Success Metric - x